# A different question from the one every other model here asks

Every notebook from [`06_linear`](06_linear.ipynb) through [`10_dl_tcn`](10_dl_tcn.ipynb) asks
the same question in different ways: **given what I can see now, what is the best guess for the
next 8-hour return?** A model that answers it well is useful whether or not any of its features
cause anything. Collinear features, proxies, coincidences - all fine, as long as the association
holds out of sample.

This notebook asks something else: **if the premium's z-score were higher, holding the other
declared drivers fixed, would the subsequent return be different?** That is a question about an
intervention, and no amount of predictive accuracy answers it. The two live in the same case
study and must not be reported as though they were the same finding, which is why the causal
result is registered under its own identity and never enters the population that
[`13_backtest`](13_backtest.ipynb) selects from.

## The estimand, stated before anything is fitted

- **Treatment:** `premium_zscore_14d`, the premium's standing relative to its own recent range.
  Continuous, not binary - so the estimate is a slope, the change in expected return per unit of
  treatment, not a difference between two groups.
- **Outcome:** `fwd_ret_8h`, the return over the settlement interval after the decision time.
- **Confounders:** `price_vol_14d`, `funding_rate`, `premium_dev_mean_14d`. These are the
  variables declared to drive both the treatment and the outcome, and adjusting for them is the
  entire identification claim.


<!-- ML4T_ZH_BEGIN c0000-p000 -->
# 與此處其他模型不同的問題

從 [`06_linear`](06_linear.ipynb) 到 [`10_dl_tcn`](10_dl_tcn.ipynb)，每份 Notebook 都以不同方式問相同問題：**根據現在可見的資訊，接下來八小時報酬率的最佳預測是什麼？** 能答得好的模型，不論特徵是否造成任何結果，都有用。共線特徵、代理變數、巧合都可以，只要關聯性能在樣本外維持。

本 Notebook 問的是另一件事：**若溢價 z 分數提高，且其他已宣告驅動因素保持不變，後續報酬率會不同嗎？** 這是介入效果的問題，再高的預測準確度也無法回答。兩者同屬一份個案研究，卻不能當作相同發現報告。因此因果結果會以自身識別值登錄，絕不進入 [`13_backtest`](13_backtest.ipynb) 用來選擇的模型群組。

## 擬合之前先說明目標估計量

- **處置變數：**`premium_zscore_14d`，溢價相對於自身近期範圍的位置。它是連續值而非二元值，因此估計量是一條斜率，表示處置值每增加一單位，預期報酬率改變多少，而非兩組之間的差異。
- **結果變數：**`fwd_ret_8h`，決策時點之後一個結算間隔的報酬率。
- **混雜因子：**`price_vol_14d`、`funding_rate`、`premium_dev_mean_14d`。這些是已宣告會同時驅動處置變數與結果變數的變數；調整它們就是全部識別主張。
<!-- ML4T_ZH_END c0000-p000 -->

That list is short, and its shortness is the honest part of the exercise. **Double machine
learning removes confounding by variables you name.** It does nothing about one you did not, and
nothing about the possibility that the relationship runs the other way. Naming three confounders
is a claim that those three are the relevant ones; the estimate is only as good as that claim.

## What double machine learning actually does

The naive approach - regress the outcome on the treatment and the confounders together - biases
the treatment coefficient whenever the confounders enter nonlinearly, because whatever the linear
term fails to absorb leaks into the treatment. DML avoids that by splitting the problem in two:

1. Predict the **outcome** from the confounders alone, and take the residual.
2. Predict the **treatment** from the confounders alone, and take the residual.
3. Regress residual on residual. What remains is the part of the treatment the confounders do not
  explain, against the part of the outcome they do not explain.

The two nuisance predictions can be any flexible learner, because the residual-on-residual step
is what carries the causal interpretation. **Cross-fitting** is what keeps that step honest: each
observation's residual is computed by a model that did not see it, so an overfitted nuisance
model cannot manufacture a residual correlation. Here the folds are chronological with an
embargo, so the nuisance models are also never fitted on data that comes after what they predict.

## Why the placebo is the part to read carefully


<!-- ML4T_ZH_BEGIN c0000-p001 -->
清單很短，而坦承這點正是此練習的誠實之處。**雙重機器學習只能消除已指明變數造成的混雜。** 對未列出的混雜因子，以及關係可能反向發生的情況，它都無能為力。列出三個混雜因子，就等於主張這三個是相關因素；估計品質取決於這項主張是否成立。

## 雙重機器學習實際如何運作

直接把結果變數對處置變數及混雜因子一併迴歸的天真做法，若混雜因子以非線性方式作用，就會使處置係數產生偏差，因為線性項未吸收的影響會滲入處置變數。DML 將問題分成兩部分以避免此情況：

1. 只根據混雜因子預測**結果變數**，取殘差。
2. 只根據混雜因子預測**處置變數**，取殘差。
3. 將殘差對殘差做迴歸。留下的是混雜因子無法解釋的處置變數部分，相對於它們無法解釋的結果變數部分。

兩個干擾模型可使用任何彈性學習器，因為殘差對殘差的步驟承載因果解釋。**交叉擬合**確保這一步可靠：每筆觀測值的殘差都由未見過它的模型計算，因此過度擬合的干擾模型不能捏造殘差相關性。此處切分依時間順序安排並設禁入間隔，因此干擾模型也不會使用晚於預測對象的資料擬合。

## 為何安慰劑檢驗最值得仔細閱讀
<!-- ML4T_ZH_END c0000-p001 -->

A causal estimate on financial data will produce a number whether or not there is anything there,
so the refutation matters more than the point estimate. The placebo permutes the treatment and
re-estimates, and the estimate should collapse. What makes it a real test rather than a formality
is the **block size**, discussed at the table below: permuting one bar at a time destroys exactly
the serial dependence that makes the original estimate hard to get right, and a placebo that
destroys the difficulty is a test the estimate passes for free.

**Learning objectives.** By the end of this notebook you will be able to:

- State an estimand - treatment, outcome, confounders, horizon - precisely enough that someone
  else could disagree with it.
- Explain what cross-fitting protects against, and why chronological folds with an embargo are
  the right shape for it on overlapping financial labels.
- Say why a block-permutation placebo needs a block long enough to preserve serial dependence,
  and how the block size is derived here rather than chosen.
- Keep a causal estimate and a predictive result in separate reports, and say in one sentence
  why combining them would misdescribe both.

**Book reference:** Chapter 15, causal inference for trading research.

**Prerequisites:** [`03_financial_features`](03_financial_features.ipynb),
[`02_labels`](02_labels.ipynb) and [`05_evaluation`](05_evaluation.ipynb) - the features, the
return labels and the purged walk-forward folds.

**What it writes:** one causal result under its own identity in `run_log/registry.db`. It is read
by [`12_model_analysis`](12_model_analysis.ipynb), which reports it **beside** the predictive
results rather than among them.

<!-- ML4T_ZH_BEGIN c0000-p002 -->
金融資料上的因果估計，無論是否存在真實效果都會產生數字，因此反駁檢驗比點估計更重要。安慰劑檢驗將處置變數排列後重新估計，估計效果應縮至接近零。使它成為真正檢驗而非形式的關鍵是**區塊大小**，下方表格會討論：若逐根 K 線排列，就會破壞原始估計之所以困難的序列相依性；消除困難的安慰劑檢驗，估計結果很容易通過。

**學習目標。** 完成本 Notebook 後，你將能夠：

- 精確說明目標估計量，包括處置、結果、混雜因子與預測期間，使其他人能對其提出不同意見。
- 解釋交叉擬合防範什麼，以及為何金融重疊標籤適合採用依時間排序且有禁入間隔的切分。
- 說明為何區塊排列安慰劑檢驗需要足夠長的區塊以保留序列相依性，以及此處如何推導而非任意選取區塊大小。
- 將因果估計與預測結果分開報告，並用一句話說明若合併會如何誤述兩者。

**書籍參考：**第 15 章，交易研究中的因果推論。

**先備知識：**[`03_financial_features`](03_financial_features.ipynb)、[`02_labels`](02_labels.ipynb) 及 [`05_evaluation`](05_evaluation.ipynb)：特徵、報酬率標籤及已清除重疊的滾動向前切分。

**寫入內容：**一項以自身識別值登錄在 `run_log/registry.db` 的因果結果。[`12_model_analysis`](12_model_analysis.ipynb) 會讀取它，將其報告在預測結果**旁邊**，而非混入預測結果之中。
<!-- ML4T_ZH_END c0000-p002 -->


In [1]:
import os

import polars as pl

from case_studies.crypto_perps_funding.research_workflow import open_study
from case_studies.research import causal_supersedes

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE = os.environ.get("ML4T_OUTPUT_DIR", "")
LABEL = "fwd_ret_8h"
CONFIG_NAME = "dml"
PREVIEW_REDUCTIONS = {}
OVERRIDES = {}
# The causal identity this run retires. A reader's clone has nothing to retire - `run_log/` is
# not shipped, so a first run meets an empty registry - and `causal_supersedes` withholds the
# declaration against a registry that does not hold it, which is why one committed value is
# right for both. A predecessor named against a registry that *does* hold rows but not this one
# is rejected at the registering write, after the DML fit and every placebo refit are paid for.
#
# The source component of a causal identity is `CAUSAL_RUNNER_VERSION`, a declared integer in
# `case_studies/utils/causal.py`; nothing hashes the file. So an edit to the estimator moves no
# identity unless that constant is raised by hand, and an edit that changes a registered value
# without raising it leaves the next run to hit the cache and serve the old number under new
# code. Once it is raised, `CausalResult.one` resolves a label to exactly one canonical
# identity, so a re-run that does not name what it replaces leaves two live and the next
# notebook fails with "resolved to 2 identities".
# Retired by this run: the block-permutation refutation now compares the HAC t-statistic
# rather than the raw effect, so CAUSAL_RUNNER_VERSION moved and every causal identity with
# it. The rows named here hold a p-value computed on the shrunken placebo effects; this run
# supersedes them rather than correcting them, because the statistic is different, not the
# arithmetic. Read out of each registry's current canonical identity per label, 2026-09-10.
SUPERSEDES_CAUSAL: str = "025f6c2f4e3b"

## 1. Resolve the estimand and the refutation contract

Nothing is fitted below. The request resolves to a specification and an identity, and the table
prints the fields that decide what the estimate means: what is being intervened on, what is being
measured, over what horizon, how the nuisance folds are cut, and how the placebo will be built.
Reading them before the fit is the only point at which disagreeing with the estimand is cheap.

`eligible_rows` is the analysis population - the rows that survive having a treatment, an
outcome and every confounder present. Quote that, not the panel height, when describing what the
estimate rests on.

<!-- ML4T_ZH_BEGIN c0003-p000 -->
## 1. 解析目標估計量與反駁檢驗契約

下方不擬合任何模型。請求解析成規格及識別值，表格印出決定估計量含義的欄位：要對什麼施加介入、衡量什麼結果、跨多長期間、如何切分干擾模型的資料，以及如何建構安慰劑檢驗。擬合前是對目標估計量提出異議成本最低的時點。

`eligible_rows` 是分析母體，也就是處置變數、結果變數及每個混雜因子都存在的資料列。描述估計所依據的資料量時，應引用這個數量，而非面板高度。
<!-- ML4T_ZH_END c0003-p000 -->


In [3]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
request = study.causal(
    method="dml",
    label=LABEL,
    config_name=CONFIG_NAME,
    execution_tier=EXECUTION_TIER,
    preview_reductions=PREVIEW_REDUCTIONS,
    overrides=OVERRIDES,
    supersedes=causal_supersedes(
        study, SUPERSEDES_CAUSAL, LABEL, labels=[LABEL], execution_tier=EXECUTION_TIER
    ),
)
resolved = request.resolve()
computation = resolved.spec["computation"]
pl.DataFrame(
    {
        "causal_hash": [resolved.identity],
        "outcome": [computation["estimand"]["outcome"]],
        "treatment": [computation["estimand"]["treatment"]],
        "outcome_horizon": [computation["estimand"]["outcome_horizon"]],
        "n_folds": [computation["cv"]["n_folds"]],
        "embargo_periods": [computation["cv"]["embargo_periods"]],
        "block_size": [computation["refutation"]["block_size"]],
        "block_size_basis": [computation["refutation"]["block_size_basis"]],
        "gap_policy": [computation["refutation"]["temporal_gap_policy"]],
        "eligible_rows": [computation["analysis_population"]["n_rows"]],
    }
)

causal_hash,outcome,treatment,outcome_horizon,n_folds,embargo_periods,block_size,block_size_basis,gap_policy,eligible_rows
str,str,str,str,i64,i64,i64,str,str,i64
"""6d6ef40064b1""","""fwd_ret_8h""","""premium_zscore_14d""","""0 days 08:00:00""",5,1,42,"""treatment_window""","""reset""",58975


`block_size` is the parameter the refutation lives or dies on, so it is on the
table rather than buried in the spec. The placebo permutes contiguous blocks
within each symbol; a block of one bar is an iid shuffle, which destroys the
serial dependence the placebo is meant to keep and makes the test trivially
easy to pass. Two things create that dependence and the block spans the longer
of them: the overlapping labels span the outcome horizon, and the treatment
spans its own construction window. Here the horizon is a single 8-hour bar
while `premium_zscore_14d` is a 42-bar rolling statistic, so the treatment
window sets the block and `block_size_basis` says so.

<!-- ML4T_ZH_BEGIN c0005-p000 -->
`block_size` 是反駁檢驗能否成立的關鍵參數，因此列在表格中，而非埋在規格裡。安慰劑檢驗在各交易代碼內排列連續區塊；若每區塊只有一根 K 線，就相當於獨立同分布的隨機重排，會破壞檢驗原本要保留的序列相依性，使檢驗輕易通過。這種相依性有兩個來源，區塊長度要涵蓋較長者：重疊標籤跨越結果預測期間，處置變數則跨越自身的建構視窗。此處結果預測期間只有一根八小時 K 線，但 `premium_zscore_14d` 是 42 根 K 線的滾動統計量，因此由處置視窗決定區塊長度，`block_size_basis` 也如此說明。
<!-- ML4T_ZH_END c0005-p000 -->


## 2. Execute, and register the result separately

The fit runs the nuisance models across the chronological folds, forms the residual-on-residual
estimate, and then pays for the placebo refits - which is where most of the cost is, since the
whole procedure is repeated once per placebo draw.

The check below refuses a result whose *computation* is not the one that was resolved. That is not
defensive coding: the computation carries the estimand, the fold geometry and the refutation
contract, and a result carrying a different one describes a contract that no longer exists. The
notebook downstream would then resolve the label to two live identities and stop.

It compares `spec["computation"]` and not the whole spec, because the spec also carries
`provenance` - the git commit, the platform and the package versions of the run that fitted. Those
record that run's own circumstances and are meant to differ from any later run that reads the row
back from cache. Comparing whole specs asserts that nothing has been committed since, which is not
a property of the estimate, and it makes the notebook raise on every re-run from a different
commit - the opposite of what this check is for. The second run of this notebook fits nothing and
has to say so rather than raise.

Comparing `result.hash` against `resolved.identity` would not do it either: both are
`training_hash_from_spec` of the same resolved spec, so that comparison cannot fail and would
assert nothing. Same check, same reasoning, as `etfs/12_causal_dml`.

<!-- ML4T_ZH_BEGIN c0006-p000 -->
## 2. 執行並單獨登錄結果

擬合程序會在依時間排序的切分上執行干擾模型，計算殘差對殘差估計量，接著付出安慰劑重新擬合的成本；後者占大部分計算量，因為每次安慰劑抽樣都要重複整套程序。

下方檢查會拒絕其*計算內容*與解析後請求不一致的結果。這不是多餘的防禦程式碼：計算內容承載目標估計量、切分結構與反駁檢驗契約；若結果包含不同內容，所描述的就是已不存在的契約。下游 Notebook 隨後會為同一標籤解析出兩個仍有效的識別值，因而停止。

它比較 `spec["computation"]`，而非整份規格，因為規格還包含 `provenance`：執行擬合時的 Git commit、平台及套件版本。這些記錄該次執行自身的環境，本來就可能與日後從快取讀回資料列的執行不同。若比較整份規格，就等於要求自上次以來不能有任何 commit；這不是估計量的性質，而且會使 Notebook 在不同 commit 上重新執行時每次都報錯，與檢查目的相反。第二次執行本 Notebook 應無須擬合任何模型，並且要明確回報，而非報錯。

比較 `result.hash` 與 `resolved.identity` 也無法達成目的：兩者都是相同解析後規格的 `training_hash_from_spec`，因此比較不可能失敗，也就沒有斷言任何性質。`etfs/12_causal_dml` 使用相同檢查與理由。
<!-- ML4T_ZH_END c0006-p000 -->


In [4]:
result = resolved.run()
if not result.complete:
    raise RuntimeError("causal execution is incomplete")
if result.spec["computation"] != resolved.spec["computation"]:
    raise RuntimeError("the registered causal computation differs from the resolved one")
pl.DataFrame(
    {
        "causal_hash": [result.hash],
        "n_obs": [result.metrics["n_obs"]],
        "complete": [result.complete],
        "execution_tier": [result.execution_tier],
    }
)

causal_hash,n_obs,complete,execution_tier
str,i64,bool,str
"""6d6ef40064b1""",54314,true,"""canonical"""


## Key takeaways and limitations

- **This is not a trading signal and must not be reported as one.** The estimate answers what
  would happen under an intervention on the premium z-score. The backtest in
  [`13_backtest`](13_backtest.ipynb) answers what a strategy would have earned. A number from
  here quoted as evidence for a strategy is a category error, and the separate registration
  exists to make that mistake require deliberate effort.
- **Adjustment covers three named confounders and nothing else.** DML removes bias from variables
  it is given. An omitted common cause, reverse causation from returns to the premium, or a
  confounder measured with error all survive it untouched, and no diagnostic in this notebook
  would reveal them. The identifying assumption is an argument, not an output.
- **The refutation is the load-bearing part.** A point estimate arrives whether or not there is
  anything to estimate. What distinguishes the two is whether the placebo collapses, and whether
  the placebo was built to be hard - which is what the block size decides.
- **The identity hashes the estimator's source.** Any edit to `case_studies/utils/causal.py`
  produces a different causal identity, whether or not it changes a number. That is deliberate: a
  causal claim is a claim about a procedure, and a changed procedure is a new claim until someone
  shows the estimate did not move.
- **A preview run is not a canonical result.** The reduced sample and fold counts a preview
  applies are recorded in its identity and it is barred from the canonical population, so a cheap
  run can never be mistaken for the published estimate.

<!-- ML4T_ZH_BEGIN c0008-p000 -->
## 重點摘要與限制

- **這不是交易訊號，也不應被當作交易訊號報告。** 估計量回答的是介入溢價 z 分數後可能發生什麼；[`13_backtest`](13_backtest.ipynb) 的回測回答策略原本可能賺取多少。若把此處數值引為策略證據，就是混淆不同類別的問題；分開登錄使這種錯誤必須刻意為之。
- **調整範圍只有三個具名混雜因子。** DML 只能消除輸入變數造成的偏差。被遺漏的共同原因、報酬率反向影響溢價，以及測量有誤的混雜因子，仍不受處理；本 Notebook 的任何診斷也無法揭露它們。識別假設須靠論證，並非模型輸出。
- **反駁檢驗是關鍵。** 無論是否存在真實效果，都會產生點估計。要區分兩者，須看安慰劑結果是否消失，以及安慰劑檢驗是否足夠嚴格；後者由區塊大小決定。
- **識別值包含估計器原始碼的雜湊。** 對 `case_studies/utils/causal.py` 的任何編輯，即使不改變數值，也會產生不同因果識別值。這是刻意設計：因果主張也是對程序的主張，程序改變後，除非有人證明估計量未變，否則就是一項新主張。
- **預覽執行不是標準結果。** 預覽使用的縮小樣本及切分數量會記入識別值，也不得進入標準模型群組，因此低成本執行絕不會被誤認為已發布的估計結果。
<!-- ML4T_ZH_END c0008-p000 -->
